# Project 3: Checkpoint 2

This part of the project has you: 
* Creating a train & test set 
* Constructing a regression pipeline (Selecting between linear regression and k-nn regression) 
* Performing cross-validation to either (a) select parameters (knn regression) or (b) evaluate the quality of predictors (linear regression) 
* Evaluating your model's performance on the test set. 

## Grading Criteria 

All project checkpoints are graded on the following criteria. **You must have satisfactory completeness** when this assignment is **first due**. You will know you have satisfactory completeness by having code in every code cell that attempts to answer the question and runs, and clearly labeled text cells that meaningfully attempts to answer every question in complete sentences. 

<details>

| **Criteria**            | **Excellent** | **Satisfactory** | **Revise** | **Not Assessable** |
|-------------------------|---------------|------------------|------------|---------------------|
| **Completeness**        | All required components are fully completed. The work goes beyond minimum requirements, with attention to detail and clear structure. | Most components are completed as required. Minor elements may be missing or incomplete, but the core work is present and functional. | Several key parts are missing, incomplete, or incorrect. Some progress is made, but the submission lacks important elements. | Very little or none of the required work is present. Submission is incomplete or off-topic. |
| **Justifying Choices**  | Clear and thoughtful justification is given for all major decisions. Demonstrates awareness of alternatives and logical reasoning. | Reasoning is provided for most decisions. Justifications are basic but show effort to explain choices. | Reasoning is limited, unclear, or superficial. Many decisions are unexplained or poorly justified. | No meaningful justification is provided. Decisions appear arbitrary or are copied without explanation. |
| **Coding Practices**    | Code is correct, runs without errors, and clearly applies methods or techniques taught in class. The code is well-organized, uses meaningful variable names, and is easy to follow. Comments enhance clarity. | Code mostly works and shows a clear attempt to use appropriate class-taught methods. There may be minor errors or inconsistencies, but the logic is mostly sound. Code is readable, though may lack polish or full clarity. | Code has significant issues (e.g., some of it does not run, gives incorrect output, or follows flawed logic). May not use methods from class or apply them incorrectly. Readability is poor or confusing, with unclear structure or variable names. | Code is substantially missing, does not run, or shows no understanding of class-taught concepts. May be copied, off-topic, or nonsensical. |
| **Visualization Quality (if applicable)** | Visualizations are clear, appropriate, well-labeled, and enhance understanding. Thoughtful choices of chart types and formatting. | Visualizations are generally appropriate and correctly formatted. Some labels or design choices could be improved. | Visualizations are present but may be unclear, mislabeled, or poorly chosen. Minimal effort to align visuals with data meaning. | Visualizations are missing, incorrect, or irrelevant to the task. |
| **Accurate Interpretation** | Interpretations are accurate, well-supported, and clearly communicated. Student demonstrates strong understanding of results. | Interpretations are mostly accurate and reasonable. May lack depth or precision, but show basic understanding. | Interpretations contain misunderstandings or are too vague. Some conclusions may not align with the data. | No interpretation is provided, or interpretations are clearly incorrect or misleading. |
    
</details>


### 1. Load data, Plan Ahead

a) Create a code cell, import `pandas` with the alias `pd`, then copy your saved dataframe from last week  (with any new columns) to the "data" folder and load it with a sensible name. [Coding Practices]

b) In a text cell below, recall your plans: For this regression problem, what do you plan to use as your predictors, and what do you plan to predict? **Make sure that you have ensured your set of predictors and response variables was approved by Prof. Masden - if not, you might not get full credit here and will have difficulty proceeding. For instance, you will need at least two predictors** [Completeness]
 
c) Look ahead at the instructions for the next several parts. Based on your previous visualizations of the relationship between your predictors and your response variable, do you plan to use k-nn regression or linear regression? If so, what features of the data do you think make this the better choice? Finally, what computations do you plan to do? [Justification]

In [167]:
import pandas as pd
import plotly.express as px

hospitals = pd.read_csv("data/hospitals_filtered.csv")
hospitals.drop("Unnamed: 0", axis=1, inplace=True)
hospitals.head()

,Discharge_Yr,OSHPD_ID,FACILITY_NAME,Hospital_County,Hospital_Zip,TypeCare,Adjusted_Length_of_Stay,MDC,Discharges,Valid_Charges,Average_Charges
0,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,1.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,11,171456,15586.909091
1,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,2.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,4,92128,23032.000000
2,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,3.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,3,77956,25985.333333
3,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,4.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,4,151857,37964.250000
4,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,6.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,3,139953,46651.000000


Predictors: discharge year, length of stay. Predict: average charges


I plan to use knn-regression because my data isn't clustered in a linear way. I don't think I need to change much in order to perform regression.

In [168]:
# attribution: https://simplemaps.com/data/us-counties
counties = pd.read_csv("data/uscounties.csv")
counties = counties.loc[counties["state_id"]=="CA"].reset_index(drop=True)[["county", "lat", "lng"]]
counties.head()

,county,lat,lng
0,Los Angeles,34.3219,-118.2247
1,San Diego,33.0343,-116.7350
2,Orange,33.7031,-117.7609
3,Riverside,33.7437,-115.9938
4,San Bernardino,34.8414,-116.1785


In [169]:
hospitals = hospitals.merge(counties, left_on="Hospital_County", right_on="county")
hospitals

,Discharge_Yr,OSHPD_ID,FACILITY_NAME,Hospital_County,Hospital_Zip,TypeCare,Adjusted_Length_of_Stay,MDC,Discharges,Valid_Charges,Average_Charges,county,lat,lng
0,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,1.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,11,171456,15586.909091,Alameda,37.6469,-121.8887
1,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,2.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,4,92128,23032.000000,Alameda,37.6469,-121.8887
2,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,3.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,3,77956,25985.333333,Alameda,37.6469,-121.8887
3,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,4.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,4,151857,37964.250000,Alameda,37.6469,-121.8887
4,2011,10735,ALAMEDA HOSPITAL,Alameda,94501,Acute Care,6.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,3,139953,46651.000000,Alameda,37.6469,-121.8887
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513160,2014,191227,LAC/HARBOR-UCLA MEDICAL CENTER,Los Angeles,90502,Acute Care,2.0,"RESPIRATORY SYSTEM, DISEASES & DISORDERS",233,4050694,17384.952790,Los Angeles,34.3219,-118.2247
513161,2014,191227,LAC/HARBOR-UCLA MEDICAL CENTER,Los Angeles,90502,Acute Care,3.0,"RESPIRATORY SYSTEM, DISEASES & DISORDERS",180,4521408,25118.933333,Los Angeles,34.3219,-118.2247
513162,2014,191227,LAC/HARBOR-UCLA MEDICAL CENTER,Los Angeles,90502,Acute Care,1.0,"SKIN, SUBCUTANEOUS TISSUE & BREAST, DISEASES &...",248,2195981,8854.762097,Los Angeles,34.3219,-118.2247
513163,2014,191227,LAC/HARBOR-UCLA MEDICAL CENTER,Los Angeles,90502,Acute Care,2.0,"SKIN, SUBCUTANEOUS TISSUE & BREAST, DISEASES &...",172,2841674,16521.360465,Los Angeles,34.3219,-118.2247


## Regression 

### 2. Getting Set up (Making your Pipeline)

1. Construct a training dataframe and a test dataframe from your saved dataframe. Save them as `yourname_train.csv` and `yourname_test.csv` (or similar). In all future analyses, you should load *this* train-test split. 

2. Construct your preprocessor steps. You might choose to include: 
   * Standardization (This is no longer ALWAYS necessary - consider whether you're doing linear regression vs. k-nn regression)
   * *One-hot encoding*  (Remember there is a difference between doing this for knn and linear regression)
   * *Oversampling*
   * Creating an *imputer*
   
3. Create your regressor model specification (either k-nn or linear regression)

3. Justification: Why do you expect the preprocessing steps you performed to improve your regression model?  If you did not choose to use any preprocessing steps, explain why you do not think they are necessary. Basically, explain why you did or did not choose *each* of the preprocessors above. 

4. Construct a pipeline with all appropriate pre-processing steps, ending with your regression model. 

In [170]:
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Output dataframes instead of arrays
set_config(transform_output="pandas")

In [171]:
hospital_train, hospital_test = train_test_split(hospitals,
                                                 train_size=.75,
                                                 random_state=2006
)

hospital_train.to_csv("data/emelia_train.csv")
hospital_train.to_csv("data/emelia_test.csv")

In [172]:
columns_of_interest = ["Discharge_Yr", "Adjusted_Length_of_Stay", "lat", "lng"]
X = hospital_train[columns_of_interest]
y = hospital_train["Average_Charges"]

scale = make_column_transformer((StandardScaler(), columns_of_interest))
knn = KNeighborsRegressor(n_neighbors=100)

In [173]:
#this never finishes running. curious.

# kvals = range(1, 200, 5)
# rmse_to_plot = []

# for k in kvals: 
#     knn_spec = KNeighborsRegressor(n_neighbors = k) 
    
#     pipe = make_pipeline(scale, knn_spec)
    
#     cv = pd.DataFrame(cross_validate(pipe, 
#                                     X=X, 
#                                     y=y,
#                                     cv=5, 
#                                     scoring="neg_root_mean_squared_error", 
#                                     ))
    
#     rmse = cv["test_score"].mean()
    
#     rmse_to_plot.append(rmse)

In [174]:
pipe = make_pipeline(scale, knn)

> Justification: Why do you expect the preprocessing steps you performed to improve your regression model? If you did not choose to use any preprocessing steps, explain why you do not think they are necessary. Basically, explain why you did or did not choose each of the preprocessors above. 

I chose to scale because I was using k neighbors regression. I also merged my hospital data with the county coordinates from another file so that I could use location as a variable in my regression. I haven't oversampled because I don't think it's necessary (based on what I know as of now) and I haven't one-hot encoded because I want to start small with 4 variables to see what happens. I don't have N/A values so I don't need an imputer.

In [175]:
subtrain, validation = train_test_split(hospital_train,
                                    train_size=.75,
                                    random_state=2006)

### 3. Check your code: Perform a fit and predict on one subtraining-validation set

1. Check that your pipeline's `.fit()` method works by fitting it on a subtraining set
2. Check that your pipelines `.predict()` method works by predicting on a validation set
3. Obtain the RMSPE on your training set. What does this number mean, in context? 

(At this point, if you can't get this to work -- turn this in anyway; I'll help you figure out what's going wrong. This is better than waiting to turn it in later)

In [176]:
sub_X = subtrain[["Discharge_Yr", "Adjusted_Length_of_Stay", "lat", "lng"]]
sub_y = subtrain["Average_Charges"]

fitted = pipe.fit(sub_X, sub_y)
validation["Predicted_Cost"] = fitted.predict(validation)
validation.loc[validation["Average_Charges"]<100000]

,Discharge_Yr,OSHPD_ID,FACILITY_NAME,Hospital_County,Hospital_Zip,TypeCare,Adjusted_Length_of_Stay,MDC,Discharges,Valid_Charges,Average_Charges,county,lat,lng,Predicted_Cost
240629,2012,331152,CORONA REGIONAL MEDICAL CENTER-MAIN,Riverside,92882,Skilled Nursing / Intermediate Care (includes ...,27.0,"RESPIRATORY SYSTEM, DISEASES & DISORDERS",1,77490,77490.000000,Riverside,33.7437,-115.9938,272181.147316
137565,2011,430837,O'CONNOR HOSPITAL,Santa Clara,95128,Acute Care,5.0,MENTAL DISEASES & DISORDERS,1,88061,88061.000000,Santa Clara,37.2318,-121.6951,71371.073916
272787,2012,380842,JEWISH HOME,San Francisco,94112,Skilled Nursing / Intermediate Care (includes ...,12.0,"NERVOUS SYSTEM, DISEASES & DISORDERS",1,5520,5520.000000,San Francisco,37.7556,-122.4450,133229.961406
34442,2011,190323,GLENDALE ADVENTIST MEDICAL CENTER,Los Angeles,91206,Acute Care,2.0,"PREGNANCY, CHILDBIRTH, & THE PUERPERIUM",899,17899730,19910.711902,Los Angeles,34.3219,-118.2247,25852.850863
312104,2013,40802,ORCHARD HOSPITAL,Butte,95948,Acute Care,3.0,"ENDOCRINE, NUTRITIONAL, AND METABOLIC, DISEASE...",11,194616,17692.363636,Butte,39.6669,-121.6007,39848.025227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164901,2012,74017,SAN RAMON REGIONAL MEDICAL CENTER,Contra Costa,94583,Acute Care,2.0,ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGA...,12,517698,43141.500000,Contra Costa,37.9192,-121.9275,30265.904010
247269,2012,340913,KAISER FOUNDATION HOSPITAL - SACRAMENTO,Sacramento,95825,Acute Care,27.0,MYELOPROLIFERATIVE DISEASES & POORLY DIFFERENT...,1,0,0.000000,Sacramento,38.4493,-121.3443,285719.772440
295177,2012,454012,VIBRA HOSPITAL OF NORTHERN CALIFORNIA,Shasta,96001,Acute Care,11.0,"INJURIES, POISONINGS, AND TOXIC EFFECTS OF DRUGS",1,29743,29743.000000,Shasta,40.7637,-122.0405,114138.698797
44107,2011,190521,MEMORIAL HOSPITAL OF GARDENA,Los Angeles,90247,Acute Care,3.0,"EYE, DISEASES & DISORDERS",1,17396,17396.000000,Los Angeles,34.3219,-118.2247,31243.319940


In [177]:
px.scatter(validation.sample(20000, random_state=2026),
          x="Predicted_Cost",
          y="Average_Charges")

<!doctype html>

In [178]:
fitted.score(X=sub_X, y=sub_y, scoring="neg_root_mean_squared_error")

0.38497165904435104

In [179]:
import numpy as np

validation["Predicted_Cost"] = pipe.predict(validation[columns_of_interest])
sq_err = (validation["Predicted_Cost"]-validation["Average_Charges"])**2
error=np.sqrt(sq_err.mean())

print("Validation error:", error)

Validation error: 338464.6925748066


In [180]:
sample = validation.sample(20000, random_state=2026)
sample["larger_than_standard_error"] = ((sample["Predicted_Cost"]-sample["Average_Charges"])>error)|((sample["Average_Charges"]-sample["Predicted_Cost"])>error)

px.scatter(sample,
          y="Adjusted_Length_of_Stay",
          x="Predicted_Cost",
          color="larger_than_standard_error",
          hover_data="Average_Charges")

<!doctype html>

I think my predictor must be really terrible but I can't tell why. I should probably take into account the type of medical care someone is receiving because I'm sure that heavily influences the cost. My predictor is off by an average of ~$300,000. That's _huge_.

### 5. For Excellent: Visualize the predictions of your model. 

Using examples from class and lab, visualize the predictions of your model when plotting one of your predictors as an x-variable and the model's predictions as a y-variable. (If you have more than one predictor, pick one to use as the x-variable, and use the average value of the other one(s) as the y-variable; alternately, if one of your predictors is categorical, you can do as we did in lab and use it as a color).

In [181]:
px.scatter(sample,
          x="Predicted_Cost",
          y="Average_Charges",
          color="Hospital_County")

<!doctype html>

In [182]:
px.scatter(sample,
          x="Predicted_Cost",
          y="Average_Charges",
          color="larger_than_standard_error")

<!doctype html>

In [183]:
px.scatter(sample,
          x="Predicted_Cost",
          y="Average_Charges",
          color="MDC").update_layout(showlegend=False)

<!doctype html>

In [184]:
# I think I'm also going to try a different k value
knn=KNeighborsRegressor(n_neighbors=11)

pipe2=make_pipeline(scale, knn)
fit_this = pipe2.fit(sub_X, sub_y)

sample["take2"]=fit_this.predict(sample)

In [185]:
fit_this.score(X=sub_X, y=sub_y, scoring="neg_root_mean_squared_error")

0.43330297652325034

In [186]:
sq_err = (sample["take2"]-sample["Average_Charges"])**2
new_error=np.sqrt(sq_err.mean())

print("Validation error:", new_error) #that's 30k better but what's wrong with my model . . . 

Validation error: 354858.8024977777


In [187]:
chargestd = sample["Average_Charges"].std()
chargestd

np.float64(450229.1589938737)

My predictor is better than the naive estimate.

In [188]:
sample["larger_than_std"] = ((sample["take2"]-sample["Average_Charges"])>chargestd)|((sample["Average_Charges"]-sample["take2"])>chargestd)

px.scatter(sample,
          x="take2",
          y="Average_Charges",
          color="larger_than_std")

<!doctype html>

In [189]:
# it might be good to pick only a few diagnostic categories intially 
hospitals.value_counts("MDC")

MDC
RESPIRATORY SYSTEM, DISEASES & DISORDERS                             41232
INFECTIOUS & PARASITIC DISEASES                                      40505
NERVOUS SYSTEM, DISEASES & DISORDERS                                 33414
CIRCULATORY SYSTEM, DISEASES & DISORDERS                             33273
DIGESTIVE SYSTEM, DISEASES & DISORDERS                               32765
MENTAL DISEASES & DISORDERS                                          29437
NEWBORNS AND NEONATE CONDITIONS BEGAN IN PERINATAL PERIOD            28816
MUSCULOSKELETAL SYSTEM & CONNECTIVE TISSUE, DISEASES & DISORDERS     27692
FACTORS ON HEALTH STATUS & OTHER CONTACTS WITH HEALTH SERVICES       25080
KIDNEY AND URINARY TRACT, DISEASES & DISORDERS                       23765
HEPATOBILIARY SYSTEM & PANCREAS, DISEASES & DISORDERS                23593
SKIN, SUBCUTANEOUS TISSUE & BREAST, DISEASES & DISORDERS             20867
ENDOCRINE, NUTRITIONAL, AND METABOLIC, DISEASES & DISORDERS          19641
MYELOPROLIFERATIVE DI

In [190]:
hospitals.value_counts("TypeCare")

TypeCare
Acute Care                                                                457390
Skilled Nursing / Intermediate Care (includes GAC approved swing beds)     23433
Psychiatric Care                                                           22040
Physical Rehabilitation Care                                                8061
Chemical Dependency Recovery Care                                           2234
Invalid/Blank                                                                  7
Name: count, dtype: int64

In [191]:
from sklearn.preprocessing import OneHotEncoder

one_hot = make_column_transformer((OneHotEncoder(sparse_output=False, handle_unknown="ignore"),["MDC", "TypeCare"]),remainder="passthrough", verbose_feature_names_out=False)

In [192]:
test = one_hot.fit_transform(subtrain)
test.columns

Index(['MDC_ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGANIC MENTAL DISEASES',
       'MDC_BLOOD, BLOOD FORMING ORGANS,IMMUNOLOGICAL, DISEASES & DISORDERS',
       'MDC_BURNS', 'MDC_CIRCULATORY SYSTEM, DISEASES & DISORDERS',
       'MDC_DIGESTIVE SYSTEM, DISEASES & DISORDERS',
       'MDC_EAR, NOSE, MOUTH, & THROAT, DISEASES & DISORDERS',
       'MDC_ENDOCRINE, NUTRITIONAL, AND METABOLIC, DISEASES & DISORDERS',
       'MDC_EYE, DISEASES & DISORDERS',
       'MDC_FACTORS ON HEALTH STATUS & OTHER CONTACTS WITH HEALTH SERVICES',
       'MDC_FEMALE REPRODUCTIVE SYSTEM, DISEASES & DISORDERS',
       'MDC_HEPATOBILIARY SYSTEM & PANCREAS, DISEASES & DISORDERS',
       'MDC_HUMAN IMMUNODEFICIENCY VIRUS INFECTIONS',
       'MDC_INFECTIOUS & PARASITIC DISEASES',
       'MDC_INJURIES, POISONINGS, AND TOXIC EFFECTS OF DRUGS',
       'MDC_KIDNEY AND URINARY TRACT, DISEASES & DISORDERS',
       'MDC_MALE REPRODUCTIVE SYSTEM, DISEASES & DISORDERS',
       'MDC_MENTAL DISEASES & DISORDERS', 'MDC_MULT

In [193]:
predict_columns = ['MDC_ALCOHOL-DRUG USE AND ALCOHOL-DRUG INDUCED ORGANIC MENTAL DISEASES',
       'MDC_BLOOD, BLOOD FORMING ORGANS,IMMUNOLOGICAL, DISEASES & DISORDERS',
       'MDC_BURNS', 'MDC_CIRCULATORY SYSTEM, DISEASES & DISORDERS',
       'MDC_DIGESTIVE SYSTEM, DISEASES & DISORDERS',
       'MDC_EAR, NOSE, MOUTH, & THROAT, DISEASES & DISORDERS',
       'MDC_ENDOCRINE, NUTRITIONAL, AND METABOLIC, DISEASES & DISORDERS',
       'MDC_EYE, DISEASES & DISORDERS',
       'MDC_FACTORS ON HEALTH STATUS & OTHER CONTACTS WITH HEALTH SERVICES',
       'MDC_FEMALE REPRODUCTIVE SYSTEM, DISEASES & DISORDERS',
       'MDC_HEPATOBILIARY SYSTEM & PANCREAS, DISEASES & DISORDERS',
       'MDC_HUMAN IMMUNODEFICIENCY VIRUS INFECTIONS',
       'MDC_INFECTIOUS & PARASITIC DISEASES',
       'MDC_INJURIES, POISONINGS, AND TOXIC EFFECTS OF DRUGS',
       'MDC_KIDNEY AND URINARY TRACT, DISEASES & DISORDERS',
       'MDC_MALE REPRODUCTIVE SYSTEM, DISEASES & DISORDERS',
       'MDC_MENTAL DISEASES & DISORDERS', 'MDC_MULTIPLE SIGNFICANT TRAUMA',
       'MDC_MUSCULOSKELETAL SYSTEM & CONNECTIVE TISSUE, DISEASES & DISORDERS',
       'MDC_MYELOPROLIFERATIVE DISEASES & POORLY DIFFERENTIATED NEOPLASMS',
       'MDC_NERVOUS SYSTEM, DISEASES & DISORDERS',
       'MDC_NEWBORNS AND NEONATE CONDITIONS BEGAN IN PERINATAL PERIOD',
       'MDC_PREGNANCY, CHILDBIRTH, & THE PUERPERIUM',
       'MDC_RESPIRATORY SYSTEM, DISEASES & DISORDERS',
       'MDC_SKIN, SUBCUTANEOUS TISSUE & BREAST, DISEASES & DISORDERS',
       'MDC_UNGROUPABLE', 'TypeCare_Acute Care',
       'TypeCare_Chemical Dependency Recovery Care', 'TypeCare_Invalid/Blank',
       'TypeCare_Physical Rehabilitation Care', 'TypeCare_Psychiatric Care',
       'TypeCare_Skilled Nursing / Intermediate Care (includes GAC approved swing beds)',
       'Discharge_Yr', 'Adjusted_Length_of_Stay', 'lat', 'lng']

big_scaler = make_column_transformer((StandardScaler(), predict_columns))

final_pipe = make_pipeline(one_hot, big_scaler, knn)

fit3 = final_pipe.fit(subtrain[["MDC", "TypeCare", "Discharge_Yr", "Adjusted_Length_of_Stay", "lat", "lng"]], sub_y)

In [194]:
sample["onelasttime"] = fit3.predict(sample)

In [195]:
sq_err = (sample["onelasttime"]-sample["Average_Charges"])**2
new_error=np.sqrt(sq_err.mean())

print("Validation error:", new_error) #that's so so so much better! wow!!

Validation error: 287955.0530244672


In [196]:
sample["larger_than_fit3"] = ((sample["onelasttime"]-sample["Average_Charges"])>new_error)|((sample["Average_Charges"]-sample["onelasttime"])>new_error)

px.scatter(sample,
          x="onelasttime",
          y="Average_Charges",
          color="larger_than_fit3")

<!doctype html>

In [197]:
px.scatter(sample,
          y="Adjusted_Length_of_Stay",
          x="onelasttime",
          color="larger_than_fit3",
          hover_data="Average_Charges") # it's still very bad here

<!doctype html>

It's getting so close ($200,000 is not close)!!